In [3]:
# ============================================================
# ScamShield AI — Notebook 3: URL Phishing Detector
# ============================================================
# 
# URLs have structural properties that reveal phishing:
# We extract 20+ features from each URL.
#
# WHY FEATURES INSTEAD OF RAW TEXT?
# A URL like "http://paypa1.com/login" is not natural text.
# TF-IDF won't understand that "paypa1" is suspicious.
# But feature engineering CAN capture:
#   - "1 instead of l" in brand names
#   - Excessive subdomains
#   - IP addresses used as domains
#   - Suspicious TLDs
# ============================================================

import os
import re
import pickle
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse
import tldextract
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

# Paths
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'backend', 'saved_models')

plt.style.use('seaborn-v0_8-darkgrid')

print("=" * 55)
print("  ScamShield AI — URL Phishing Detector")
print("=" * 55)


# ============================================================
# URL FEATURE EXTRACTION FUNCTION
# ============================================================
# This is the HEART of our URL classifier.
# We extract 25 handcrafted features from each URL.

# Lists of suspicious and legitimate patterns
SUSPICIOUS_TLDS = {
    'xyz', 'tk', 'ml', 'ga', 'cf', 'gq', 'pw', 'top',
    'loan', 'click', 'download', 'racing', 'review',
    'country', 'stream', 'gdn', 'win', 'bid'
}

LEGITIMATE_BRANDS = [
    'google', 'facebook', 'amazon', 'paypal', 'apple',
    'microsoft', 'netflix', 'instagram', 'twitter',
    'linkedin', 'github', 'sbi', 'hdfc', 'icici', 
    'paytm', 'phonepe', 'gpay', 'flipkart', 'myntra',
    'swiggy', 'zomato', 'irctc', 'uidai', 'gov'
]

URGENCY_WORDS_IN_URL = [
    'verify', 'secure', 'update', 'confirm', 'login',
    'signin', 'account', 'suspend', 'block', 'validate',
    'urgent', 'alert', 'warning', 'critical', 'prize',
    'winner', 'claim', 'free', 'lucky', 'kyc'
]


def extract_url_features(url):
    """
    Extract 25 structural and semantic features from a URL.
    
    These features capture patterns that distinguish
    phishing URLs from legitimate ones.
    
    Args:
        url: String URL
        
    Returns:
        dict: 25 feature key-value pairs
    """
    features = {}
    
    # ── Safety: handle malformed URLs ────────────────────
    try:
        parsed = urlparse(url)
        extracted = tldextract.extract(url)
    except Exception:
        # Return all-zero features for completely invalid URLs
        return {f'feature_{i}': 0 for i in range(25)}
    
    # Get URL components
    scheme    = parsed.scheme.lower()       # http or https
    domain    = extracted.domain.lower()    # e.g., 'google'
    suffix    = extracted.suffix.lower()    # e.g., 'com'
    subdomain = extracted.subdomain.lower() # e.g., 'www'
    path      = parsed.path.lower()
    query     = parsed.query.lower()
    full_url  = url.lower()
    
    # ── GROUP 1: Length Features ──────────────────────────
    # Phishing URLs tend to be longer (to look complex/official)
    features['url_length'] = len(url)
    features['domain_length'] = len(domain)
    features['path_length'] = len(path)
    features['subdomain_length'] = len(subdomain)
    
    # ── GROUP 2: Security Features ────────────────────────
    # Legitimate sites use HTTPS; phishing often use HTTP
    features['uses_https'] = 1 if scheme == 'https' else 0
    features['uses_http']  = 1 if scheme == 'http' else 0
    
    # IP address as domain = very suspicious
    ip_pattern = r'^(\d{1,3}\.){3}\d{1,3}$'
    features['has_ip_address'] = 1 if re.match(ip_pattern, parsed.netloc) else 0
    
    # ── GROUP 3: Count Features ───────────────────────────
    features['dot_count']      = url.count('.')
    features['dash_count']     = url.count('-')
    features['slash_count']    = url.count('/')
    features['at_count']       = url.count('@')  # @ in URL = redirect trick
    features['question_count'] = url.count('?')
    features['equals_count']   = url.count('=')
    features['underscore_count'] = url.count('_')
    features['digit_count']    = sum(c.isdigit() for c in url)
    
    # Ratio of digits to total length
    features['digit_ratio'] = features['digit_count'] / max(len(url), 1)
    
    # ── GROUP 4: Subdomain Features ───────────────────────
    # Phishing: secure.banking.sbi.fake-site.com
    # Legitimate: www.sbi.co.in
    subdomains = [s for s in subdomain.split('.') if s and s != 'www']
    features['subdomain_count'] = len(subdomains)
    features['has_multiple_subdomains'] = 1 if len(subdomains) >= 2 else 0
    
    # ── GROUP 5: TLD Features ─────────────────────────────
    features['is_suspicious_tld'] = 1 if suffix in SUSPICIOUS_TLDS else 0
    features['is_gov_domain'] = (
    1 if suffix.endswith('.gov.in') or suffix == 'gov' else 0
)
    
    # ── GROUP 6: Brand Impersonation ──────────────────────
    # Check if URL contains brand name but domain is NOT the brand
    brand_in_url = any(brand in full_url for brand in LEGITIMATE_BRANDS)
    brand_is_domain = any(brand == domain for brand in LEGITIMATE_BRANDS)
    
    # Suspicious: brand in URL but not as the actual domain
    # e.g., "paypal-secure.xyz" contains "paypal" but domain is "paypal-secure"
    features['brand_in_subdomain_or_path'] = (
        1 if (brand_in_url and not brand_is_domain) else 0
    )
    
    # ── GROUP 7: Urgency/Scam Words in URL ────────────────
    urgency_count = sum(1 for word in URGENCY_WORDS_IN_URL 
                       if word in full_url)
    features['urgency_word_count'] = urgency_count
    features['has_urgency_words']  = 1 if urgency_count > 0 else 0
    
    # ── GROUP 8: Special Patterns ─────────────────────────
    # URL shorteners hide destination (suspicious)
    shorteners = ['bit.ly', 'tinyurl', 'goo.gl', 't.co', 
                  'ow.ly', 'short.io', 'tiny.cc']
    features['is_url_shortener'] = 1 if any(s in full_url 
                                             for s in shorteners) else 0
    
    # Hex encoding in URL (obfuscation technique)
    features['has_hex_encoding'] = 1 if '%' in url else 0
    
    return features


# Test the function
test_urls = [
    'https://www.sbi.co.in/web/personal-banking',
    'http://sbi-secure-verify.xyz/login/confirm',
    'http://192.168.1.1/bank/login.php',
    'http://secure.banking.paypal.fake-site.ml'
]

print("Feature extraction test:")
print()
for test_url in test_urls:
    feats = extract_url_features(test_url)
    print(f"URL: {test_url[:60]}")
    suspicious_feats = {k: v for k, v in feats.items() if v > 0 and k != 'url_length'}
    print(f"  Suspicious signals: {suspicious_feats}")
    print()

print(f"Total features per URL: {len(extract_url_features(test_urls[0]))}")

  ScamShield AI — URL Phishing Detector
Feature extraction test:

URL: https://www.sbi.co.in/web/personal-banking
  Suspicious signals: {'domain_length': 3, 'path_length': 21, 'subdomain_length': 3, 'uses_https': 1, 'dot_count': 3, 'dash_count': 1, 'slash_count': 4}

URL: http://sbi-secure-verify.xyz/login/confirm
  Suspicious signals: {'domain_length': 17, 'path_length': 14, 'uses_http': 1, 'dot_count': 1, 'dash_count': 2, 'slash_count': 4, 'is_suspicious_tld': 1, 'brand_in_subdomain_or_path': 1, 'urgency_word_count': 4, 'has_urgency_words': 1}

URL: http://192.168.1.1/bank/login.php
  Suspicious signals: {'domain_length': 11, 'path_length': 15, 'uses_http': 1, 'has_ip_address': 1, 'dot_count': 4, 'slash_count': 4, 'digit_count': 8, 'digit_ratio': 0.24242424242424243, 'urgency_word_count': 1, 'has_urgency_words': 1}

URL: http://secure.banking.paypal.fake-site.ml
  Suspicious signals: {'domain_length': 9, 'subdomain_length': 21, 'uses_http': 1, 'dot_count': 4, 'dash_count': 1, 'slash_

In [4]:
# ============================================================
# BUILD FEATURE MATRICES FOR ALL URLs
# ============================================================

print("Loading URL datasets...")

# Load train and test URL data
url_train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'url_train.csv'))
url_test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, 'url_test.csv'))

print(f"  Train URLs: {len(url_train_df)}")
print(f"  Test URLs:  {len(url_test_df)}")
print()
print("Extracting features from training URLs...")

# Extract features for all training URLs
train_features = []
for url in url_train_df['url']:
    train_features.append(extract_url_features(str(url)))

# Extract features for all test URLs
test_features = []
for url in url_test_df['url']:
    test_features.append(extract_url_features(str(url)))

# Convert to DataFrames
train_features_df = pd.DataFrame(train_features)
test_features_df  = pd.DataFrame(test_features)

# Get labels
y_train_url = url_train_df['label'].values.astype(int)
y_test_url  = url_test_df['label'].values.astype(int)

X_train_url = train_features_df.values
X_test_url  = test_features_df.values

print(f"Feature matrix shape:")
print(f"  Train: {X_train_url.shape}")
print(f"  Test:  {X_test_url.shape}")
print()

# Show feature statistics
print("Feature statistics (training set):")
stats = train_features_df.describe().round(3)
print(stats.T[['mean', 'std', 'min', 'max']].to_string())

Loading URL datasets...
  Train URLs: 32
  Test URLs:  9

Extracting features from training URLs...
Feature matrix shape:
  Train: (32, 25)
  Test:  (9, 25)

Feature statistics (training set):
                              mean     std   min      max
url_length                  41.344  16.275  29.0  125.000
domain_length               11.906  10.431   3.0   59.000
path_length                 12.094   5.959   0.0   22.000
subdomain_length             2.750   6.011   0.0   27.000
uses_https                   0.438   0.504   0.0    1.000
uses_http                    0.562   0.504   0.0    1.000
has_ip_address               0.062   0.246   0.0    1.000
dot_count                    1.844   1.081   1.0    5.000
dash_count                   1.188   1.655   0.0    7.000
slash_count                  3.594   0.837   2.0    6.000
at_count                     0.000   0.000   0.0    0.000
question_count               0.125   0.336   0.0    1.000
equals_count                 0.188   0.592   0.0    3

In [5]:
# ============================================================
# TRAIN URL CLASSIFIERS
# ============================================================

url_model_results = []
cv_url = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ── Model 1: Random Forest ────────────────────────────────
print("Training Random Forest...")
rf_url = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_url.fit(X_train_url, y_train_url)

# Evaluate
y_pred_rf = rf_url.predict(X_test_url)
y_prob_rf  = rf_url.predict_proba(X_test_url)[:, 1]

rf_url_results = {
    'model_name': 'Random Forest (URL)',
    'accuracy':   accuracy_score(y_test_url, y_pred_rf),
    'precision':  precision_score(y_test_url, y_pred_rf, zero_division=0),
    'recall':     recall_score(y_test_url, y_pred_rf, zero_division=0),
    'f1_score':   f1_score(y_test_url, y_pred_rf, zero_division=0),
    'roc_auc':    roc_auc_score(y_test_url, y_prob_rf) if len(np.unique(y_test_url)) > 1 else 0.5,
}
url_model_results.append(rf_url_results)

print(f"  Accuracy:  {rf_url_results['accuracy']:.4f}")
print(f"  Recall:    {rf_url_results['recall']:.4f}")
print(f"  F1 Score:  {rf_url_results['f1_score']:.4f}")
print()

# ── Model 2: XGBoost ─────────────────────────────────────
print("Training XGBoost...")
neg = (y_train_url == 0).sum()
pos = (y_train_url == 1).sum()
sw  = neg / pos if pos > 0 else 1

xgb_url = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=sw,
    subsample=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1
)
xgb_url.fit(X_train_url, y_train_url)

y_pred_xgb = xgb_url.predict(X_test_url)
y_prob_xgb  = xgb_url.predict_proba(X_test_url)[:, 1]

xgb_url_results = {
    'model_name': 'XGBoost (URL)',
    'accuracy':   accuracy_score(y_test_url, y_pred_xgb),
    'precision':  precision_score(y_test_url, y_pred_xgb, zero_division=0),
    'recall':     recall_score(y_test_url, y_pred_xgb, zero_division=0),
    'f1_score':   f1_score(y_test_url, y_pred_xgb, zero_division=0),
    'roc_auc':    roc_auc_score(y_test_url, y_prob_xgb) if len(np.unique(y_test_url)) > 1 else 0.5,
}
url_model_results.append(xgb_url_results)

print(f"  Accuracy:  {xgb_url_results['accuracy']:.4f}")
print(f"  Recall:    {xgb_url_results['recall']:.4f}")
print(f"  F1 Score:  {xgb_url_results['f1_score']:.4f}")
print()

# ── Feature Importance ────────────────────────────────────
feature_cols = list(train_features_df.columns)
importance   = xgb_url.feature_importances_
sorted_idx   = np.argsort(importance)[::-1]

print("Top 15 URL Features by Importance:")
for i in sorted_idx[:15]:
    bar = '█' * int(importance[i] * 100)
    print(f"  {feature_cols[i]:<35} {bar} ({importance[i]:.4f})")

# ── Visualization ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('URL Phishing Detector — Feature Importance', fontweight='bold')

# Bar chart of top features
top_n = 15
top_features = [feature_cols[i] for i in sorted_idx[:top_n]]
top_scores   = [importance[i] for i in sorted_idx[:top_n]]

colors = ['#e74c3c' if score > 0.05 else '#3498db' for score in top_scores]
axes[0].barh(range(top_n), top_scores[::-1], color=colors[::-1])
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_features[::-1], fontsize=9)
axes[0].set_xlabel('Importance Score')
axes[0].set_title('Top URL Features\n(Red = High importance)')
axes[0].grid(True, axis='x', alpha=0.3)

# ROC Curve comparison
fpr_rf,  tpr_rf,  _ = roc_curve(y_test_url, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test_url, y_prob_xgb)

axes[1].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={rf_url_results['roc_auc']:.3f})", 
            linewidth=2, color='#3498db')
axes[1].plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={xgb_url_results['roc_auc']:.3f})", 
            linewidth=2, color='#e74c3c')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — URL Models')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save best URL model (XGBoost usually better)
best_url_model = xgb_url if xgb_url_results['f1_score'] >= rf_url_results['f1_score'] else rf_url

url_model_path = os.path.join(MODELS_DIR, 'url_classifier_xgb.pkl')
with open(url_model_path, 'wb') as f:
    pickle.dump(best_url_model, f)

# Save feature extractor info
feature_info = {
    'feature_names': feature_cols,
    'n_features': len(feature_cols),
    'suspicious_tlds': list(SUSPICIOUS_TLDS),
    'urgency_words': URGENCY_WORDS_IN_URL
}
with open(os.path.join(MODELS_DIR, 'url_feature_info.json'), 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f"\nURL model saved: {url_model_path}")
print(f"Feature info saved to model registry")

Training Random Forest...
  Accuracy:  1.0000
  Recall:    1.0000
  F1 Score:  1.0000

Training XGBoost...
  Accuracy:  1.0000
  Recall:    1.0000
  F1 Score:  1.0000

Top 15 URL Features by Importance:
  uses_https                          ██████████████████████████████████████████████████████████████████ (0.6645)
  urgency_word_count                  ████████████████ (0.1611)
  domain_length                       ██████ (0.0661)
  dot_count                           █████ (0.0553)
  dash_count                          █████ (0.0525)
  url_length                           (0.0006)
  question_count                       (0.0000)
  path_length                          (0.0000)
  subdomain_length                     (0.0000)
  uses_http                            (0.0000)
  has_ip_address                       (0.0000)
  slash_count                          (0.0000)
  at_count                             (0.0000)
  has_hex_encoding                     (0.0000)
  is_url_shortener         

<Figure size 1600x600 with 2 Axes>


URL model saved: C:\Users\Manoj S\Desktop\scamshield-ai\backend\saved_models\url_classifier_xgb.pkl
Feature info saved to model registry
